# ArXiv Paper Metadata Scraper — Report

**Course:** Web Scraping  
**Author:** NakerTheFirst  
**Target site:** [arxiv.org](https://arxiv.org)  
**Categories scraped:** `cs.AI`, `cs.CV`, `cs.LG`, `cs.CL`

---

## Contents

1. [Legal & Ethical Basis](#1-legal)
2. [Website Structure](#2-structure)
3. [Python Regex Patterns](#3-regex)
4. [Scraping Tools](#4-tools)
5. [Dataset Exploration](#5-eda)
6. [Conclusions](#6-conclusions)

In [ ]:
import sys
import inspect
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from bs4 import BeautifulSoup

plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (9, 4)})

REFERENCES = PROJECT_ROOT / 'references'
DATA_DIR   = PROJECT_ROOT / 'data'

# Pick the most recently produced master CSV
csv_files = sorted(DATA_DIR.glob('*-papers.csv'))
CSV_PATH  = csv_files[-1] if csv_files else None

print('Project root:', PROJECT_ROOT)
print('Data CSV:', CSV_PATH.name if CSV_PATH else 'not found — run python main.py first')

<a id='1-legal'></a>
## 1. Legal & Ethical Basis

ArXiv's `robots.txt` explicitly **allows** scraping of the paths used in this project:

```
User-agent: *
Crawl-delay: 15
Allow: /list
Allow: /abs
Allow: /pdf
Allow: /html
Allow: /archive
```

All scrapers in this project:
- Respect the **15-second crawl delay** between every request (`CRAWL_DELAY = 15` in `config.py`)
- Only access the allowed `/list` and `/abs` paths
- Identify themselves with a descriptive `User-Agent` header
- Do **not** download full PDFs or source archives

In [ ]:
robots = (REFERENCES / 'robots.txt').read_text(encoding='utf-8')
# Show only the default user-agent block
print(robots.split('User-agent: Googlebot')[0].strip())

<a id='2-structure'></a>
## 2. Website Structure

Two page types are scraped:

| Page | URL pattern | Content |
|------|-------------|----------|
| **Listing** | `/list/{category}/{date}` | Index: IDs, titles, authors, subjects |
| **Abstract** | `/abs/{arxiv_id}` | Full record: abstract, date, DOI, history |

### 2.1 `/list` page

Papers are wrapped in `<dl id="articles">` as `<dt>` / `<dd>` pairs.  
For **recent** listings a `<h3>` inside the `<dl>` carries the entry count:
```
Fri, 8 May 2026 (showing first 50 of 355 entries)
```
For **monthly archive** listings (`YYYY-MM`) there is no `<h3>`; the count lives in
`<div class="paging">` instead:
```
Total of 5024 entries : 1-50  51-100  ...
```
Both scrapers detect whichever format is present and paginate with
`?skip=0&show=2000`, `?skip=2000&show=2000`, … until all entries are collected.

In [ ]:
list_html = (REFERENCES / 'example_list.htm').read_text(encoding='utf-8')
list_soup = BeautifulSoup(list_html, 'html.parser')

articles = list_soup.find('dl', id='articles')
h3 = articles.find('h3')
print('Listing header:', h3.get_text(strip=True) if h3 else '(no h3 — monthly archive)')
print()

first_dt = articles.find('dt')
first_dd = articles.find('dd')

abs_anchor = first_dt.find('a', title='Abstract')
print('ArXiv ID (id attr):', abs_anchor.get('id', ''))
print('Abs href:          ', abs_anchor['href'])

title_div = first_dd.find('div', class_='list-title')
for desc in title_div.find_all('span', class_='descriptor'):
    desc.decompose()
print('Title:             ', title_div.get_text(strip=True))

authors = [a.get_text() for a in first_dd.find_all('a')]
print('Authors:           ', '; '.join(authors))

### 2.2 `/abs` page

All key fields sit inside `<div id="abs">`:

```html
<div id="abs">
  <div class="dateline">[Submitted on 7 May 2026]</div>
  <h1 class="title"><span class="descriptor">Title:</span> ...</h1>
  <div class="authors"><a>Author Name</a>, ...</div>
  <blockquote class="abstract">
    <span class="descriptor">Abstract:</span> ...
  </blockquote>
  <table>
    <tr><td class="subjects">cs.AI; cs.LG</td></tr>
    <tr><td class="comments">22 pages, 4 figures</td></tr>
  </table>
</div>
```

Each `<span class="descriptor">` ("Title:", "Abstract:") is decomposed before
calling `get_text()` so label text does not bleed into field values.

In [ ]:
abs_html = (REFERENCES / 'example_abs.htm').read_text(encoding='utf-8')
abs_soup = BeautifulSoup(abs_html, 'html.parser')
abs_div  = abs_soup.find('div', id='abs')

dateline = abs_div.find('div', class_='dateline')
print('Dateline: ', dateline.get_text(strip=True) if dateline else 'n/a')

title_tag = abs_div.find('h1', class_='title')
if title_tag:
    for d in title_tag.find_all('span', class_='descriptor'):
        d.decompose()
print('Title:    ', title_tag.get_text(strip=True) if title_tag else 'n/a')

bq = abs_div.find('blockquote', class_='abstract')
if bq:
    for d in bq.find_all('span', class_='descriptor'):
        d.decompose()
abstract = bq.get_text(strip=True) if bq else ''
print('Abstract: ', abstract[:100], '...')

subjects_td = abs_div.find('td', class_='subjects')
print('Subjects: ', subjects_td.get_text(strip=True) if subjects_td else 'n/a')

doi_link = abs_soup.find('a', id='arxiv-doi-link')
print('DOI:      ', doi_link.get_text(strip=True) if doi_link else 'n/a')

<a id='3-regex'></a>
## 3. Python Regex Patterns

All patterns are compiled once in `src/utils.py` and shared across all three scrapers.

In [ ]:
from src.utils import (
    ARXIV_ID_RE, SUBMISSION_DATE_RE, CATEGORY_CODE_RE,
    extract_arxiv_id, extract_submission_date, extract_categories,
)

# ArXiv ID
print('ArXiv ID:', ARXIV_ID_RE.pattern)
for s in ['arXiv:2605.06651v1', 'See also arXiv:2501.12345', 'no id here']:
    print(f'  {s!r:35s} → {extract_arxiv_id(s)!r}')
print()

# Submission date
print('Submission date:', SUBMISSION_DATE_RE.pattern)
for s in ['[Submitted on 7 May 2026]', '[Submitted on 12 January 2025]', '[v2] Mon, 10 Feb 2025']:
    print(f'  {s!r:40s} → {extract_submission_date(s)!r}')
print()

# Category codes
print('Category code:', CATEGORY_CODE_RE.pattern)
for s in [
    'Artificial Intelligence (cs.AI)',
    'Machine Learning (cs.LG); Statistics (stat.ML)',
]:
    primary, cross = extract_categories(s)
    print(f'  {s!r}')
    print(f'    primary={primary!r}  cross={cross}')

In [ ]:
from src.utils import (
    _LISTING_COUNT_RE, _LISTING_DATE_RE, _PAGING_TOTAL_RE,
    parse_listing_header, parse_paging_total,
)

# Recent/daily listings — count in h3
print('h3 format (recent listings):')
print('  count pattern:', _LISTING_COUNT_RE.pattern)
for h in [
    'Fri, 8 May 2026 (showing first 50 of 355 entries)',
    'Mon, 5 May 2026 (showing first 50 of 50 entries)',
]:
    date, shown, total = parse_listing_header(h)
    print(f'  {h!r}')
    print(f'    → date={date!r}  shown={shown}  total={total}  truncated={shown < total}')
print()

# Monthly archive listings — count in div.paging
print('div.paging format (monthly archives):')
print('  count pattern:', _PAGING_TOTAL_RE.pattern)
for p in [
    'Total of 5024 entries : 1-50 51-100 101-150',
    'Total of 42 entries : 1-42',
]:
    print(f'  {p!r:50s} → total={parse_paging_total(p)}')

In [ ]:
from src.utils import DOI_RE, clean_text

print('DOI pattern:', DOI_RE.pattern)
for s in [
    '10.48550/arXiv.2605.06651',
    'doi: 10.1038/s41586-021-03819-2',
    'no doi here',
]:
    m = DOI_RE.search(s)
    print(f'  {s!r:45s} → {m.group(0)!r if m else None!r}')
print()

dirty = '  AI  Co-Mathematician:\n  Accelerating\tMathematicians  '
print('Whitespace normalisation:')
print(f'  before: {dirty!r}')
print(f'  after:  {clean_text(dirty)!r}')

<a id='4-tools'></a>
## 4. Scraping Tools

### 4.1 requests + BeautifulSoup — `/list` pages

`src/requests_scraper.py` · `ListScraper`

- Fetches each category's `/list` page with `requests.Session`
- Detects truncation: checks `<h3>` for recent listings, `<div class="paging">` for monthly archives
- Paginates with `?skip=0&show=2000`, `?skip=2000&show=2000`, … (ArXiv max is `show=2000`)
- Pairs `<dt>` / `<dd>` siblings; decomposes `<span class="descriptor">` before `get_text()`
- Returns stub dicts — no abstract (that requires a visit to `/abs`)

In [ ]:
from src.requests_scraper import ListScraper
import json

scraper = ListScraper()
soup    = BeautifulSoup(list_html, 'html.parser')
papers  = scraper._parse_list_page(soup)
scraper.close()

print(f'Parsed {len(papers)} stubs from example_list.htm')
print()
print(json.dumps(papers[0], indent=2))

### 4.2 Selenium — `/abs` pages

`src/selenium_scraper.py` · `AbsScraper`

ArXiv `/abs` pages are static HTML, but Selenium is required by the course specification
to demonstrate browser-automation skills. It provides two concrete advantages:

1. **Explicit waits** — `WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.ID, 'abs')))`
   blocks until the DOM element is present, removing the need for arbitrary `time.sleep()` calls.
2. **Full rendered DOM** — `driver.page_source` captures the post-JavaScript state,
   so any dynamically injected content is automatically included.

In [ ]:
from src.selenium_scraper import AbsScraper

# Call the parse method directly on the saved example — no browser needed
dummy   = object.__new__(AbsScraper)
record  = AbsScraper._parse_abs_page(dummy, abs_soup, '2605.06651')

for key, val in record.items():
    preview = str(val)[:80] + ('...' if len(str(val)) > 80 else '')
    print(f'{key:25s}: {preview}')

### 4.3 Scrapy — full `/list → /abs` crawl

`src/scrapy_scraper/spiders/arxiv_spider.py` · `ArxivSpider`

The spider performs the same two-step crawl asynchronously, following `/abs` links
found on each listing page and yielding a plain dict per paper.

| Setting | Value | Reason |
|---------|-------|--------|
| `ROBOTSTXT_OBEY` | `True` | Automatic robots.txt enforcement |
| `DOWNLOAD_DELAY` | 15 s | Matches `Crawl-delay` |
| `RANDOMIZE_DOWNLOAD_DELAY` | `True` | Jitter ∈ [7.5 s, 22.5 s] |
| `AUTOTHROTTLE_ENABLED` | `True` | Server-latency-aware throttling |
| `CONCURRENT_REQUESTS` | 1 | Single in-flight request |

`DeduplicatePipeline` drops items whose `arxiv_id` was already seen in the run
(a paper listed under multiple categories would otherwise appear more than once).

In [ ]:
from src.scrapy_scraper.spiders.arxiv_spider import ArxivSpider

for name in ('start', 'parse_list', '_parse_entries', 'parse_abs'):
    method = getattr(ArxivSpider, name)
    sig    = inspect.signature(method)
    doc    = (inspect.getdoc(method) or '').split('\n')[0]
    print(f'def {name}{sig}')
    if doc:
        print(f'    """{doc}"""')
    print()

<a id='5-eda'></a>
## 5. Dataset Exploration

Run `python main.py` from the project root to populate `data/` before executing these cells.

In [ ]:
if CSV_PATH is None or not CSV_PATH.exists():
    raise FileNotFoundError('No papers CSV found in data/. Run  python main.py  first.')

df = pd.read_csv(CSV_PATH, dtype=str).fillna('')
print(f'File:   {CSV_PATH.name}')
print(f'Shape:  {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
df.head(3)

In [ ]:
# Field coverage — what fraction of records have each field populated
coverage = (
    df.replace('', pd.NA)
      .notna()
      .mean()
      .mul(100)
      .round(1)
      .rename('% populated')
      .to_frame()
)
coverage

In [ ]:
# Category distribution
cat_counts = df['primary_category'].value_counts()

fig, ax = plt.subplots()
cat_counts.plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Papers per primary category')
ax.set_xlabel('Category')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()
print(cat_counts.to_string())

In [ ]:
# Cross-listing prevalence
has_cross = df['cross_list_categories'].str.strip().ne('').sum()
print(f'{has_cross:,} of {len(df):,} papers ({has_cross/len(df):.1%}) have cross-list categories')

cross_exploded = (
    df['cross_list_categories']
    .str.split(';')
    .explode()
    .str.strip()
    .replace('', pd.NA)
    .dropna()
)
print('\nTop 10 cross-listed categories:')
print(cross_exploded.value_counts().head(10).to_string())

In [ ]:
# Submission date distribution
dates = pd.to_datetime(df['submission_date'], errors='coerce').dropna()

if not dates.empty:
    fig, ax = plt.subplots()
    dates.dt.date.value_counts().sort_index().plot.bar(ax=ax, color='teal', edgecolor='white')
    ax.set_title('Submission dates')
    ax.set_xlabel('Date')
    ax.set_ylabel('Papers')
    ax.xaxis.set_major_locator(ticker.MaxNLocator(10))
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No submission_date data — Selenium/Scrapy phase may not have run.')

In [ ]:
# Author count distribution
author_counts = (
    df['authors']
    .str.split(';')
    .apply(lambda x: len([a for a in x if a.strip()]))
)

fig, ax = plt.subplots()
author_counts.clip(upper=15).value_counts().sort_index().plot.bar(
    ax=ax, color='coral', edgecolor='white'
)
ax.set_title('Authors per paper (capped at 15)')
ax.set_xlabel('Number of authors')
ax.set_ylabel('Papers')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print(f'Median: {author_counts.median():.1f}   Max: {author_counts.max()}')

In [ ]:
# Abstract length (Scrapy + Selenium records only)
abstract_len = df['abstract'].str.len().replace(0, pd.NA).dropna()

if not abstract_len.empty:
    fig, ax = plt.subplots()
    abstract_len.plot.hist(bins=40, ax=ax, color='mediumseagreen', edgecolor='white')
    ax.axvline(abstract_len.median(), color='red', linestyle='--',
               label=f'Median = {abstract_len.median():.0f} chars')
    ax.set_title('Abstract length (characters)')
    ax.set_xlabel('Characters')
    ax.set_ylabel('Papers')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(abstract_len.describe().round(0).astype(int).to_string())
else:
    print('No abstract data available.')

In [ ]:
# Summary stats
print('=== Dataset summary ===')
print(f'Total rows:             {len(df):,}')
print(f'Unique arxiv IDs:       {df["arxiv_id"].nunique():,}')
print(f'Categories covered:     {df["primary_category"].nunique()}')
print(f'Records with abstract:  {(df["abstract"].str.len() > 0).sum():,}')
print(f'Records with DOI:       {(df["doi"].str.len() > 0).sum():,}')
print(f'Records with date:      {(df["submission_date"].str.len() > 0).sum():,}')

<a id='6-conclusions'></a>
## 6. Conclusions

### What was built

A three-phase pipeline demonstrating all four mandatory libraries:

| Phase | Library | Target | Fields |
|-------|---------|--------|--------|
| 1 | requests + BeautifulSoup | `/list` pages | ID, title, authors, subjects, links |
| 2 | Selenium (headless Chrome) | `/abs` pages (50-paper sample) | abstract, date, DOI, history |
| 3 | Scrapy | `/list → /abs` async crawl | all fields in one pass |

Python `re` is used throughout via six compiled patterns in `src/utils.py`:
arXiv IDs, ISO submission dates, category codes, DOIs, listing counts (two formats), and whitespace.

### Design decisions

- **Shared utilities** (`src/utils.py`) ensure consistent field extraction across all three scrapers.
- **Two-format pagination** — `<h3>` for recent listings, `<div class="paging">` for monthly archives;
  both trigger the same skip loop (`show=2000`, ArXiv's maximum).
- **Descriptor span removal** — `<span class="descriptor">Title:</span>` is decomposed before
  `get_text()` to prevent label text appearing in field values.
- **Merge priority** — Scrapy (complete records) > Selenium (enriched sample) > requests (stubs).
  `drop_duplicates(keep='first')` preserves the richest record per paper.

### Limitations

- The 15-second crawl delay limits throughput; this is a deliberate constraint imposed
  by ArXiv's `robots.txt` and is non-negotiable.
- Abstract coverage depends on how many papers the Selenium and Scrapy phases visit.
  For full abstract coverage across all scraped papers, Scrapy is the only practical option
  given the crawl delay.
- The Selenium phase scrapes a fixed 50-paper sample (`SELENIUM_SAMPLE` in `main.py`);
  its records are almost entirely superseded by Scrapy in the merge step.